In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
#another free model from  groq to demonstrate multiple subagents with different models
model2 = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model2,
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [5]:
question = "What is the square  of 12?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square  of 12?', additional_kwargs={}, response_metadata={}, id='276e4717-05e0-4c6d-850d-a64abea56eea'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2c6z9ws6d', 'function': {'arguments': '{"x":12}', 'name': 'call_subagent_2'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 339, 'total_tokens': 356, 'completion_time': 0.052016261, 'completion_tokens_details': None, 'prompt_time': 0.051264593, 'prompt_tokens_details': None, 'queue_time': 0.063028325, 'total_time': 0.103280854}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4d05-e959-7291-9f30-aee6f1048be2-0', tool_calls=[{'name': 'call_subagent_2', 'args': {'x': 12}, 'id': '2c6z9ws6d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input